In [ ]:
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
# Initializing Document Loaders
cardiomegaly_loader = PyPDFDirectoryLoader("data/Cardiomegaly.pdf")
pleuraleffusion_pneumothorax_loader = PyPDFDirectoryLoader("data/Pleural Effusion & Pneumothorax.pdf")
tuberculosis_loader = PyPDFDirectoryLoader("data/Tuberculosis_Guidelines.pdf")
xraydictionary_loader = PyPDFDirectoryLoader("data/XRay_Dictionary.pdf")

In [ ]:
# Data Ingestion
import asyncio

# Loading Documents Asynchronously (Concurrent)
cardiomegaly_docs, pleuraleffusion_pneumothorax_docs, tuberculosis_docs, xraydictionary_docs = await asyncio.gather(
    cardiomegaly_loader.aload(),
    pleuraleffusion_pneumothorax_loader.aload(),
    tuberculosis_loader.aload(),
    xraydictionary_loader.aload()
)

In [ ]:
# Text Splitting
text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)

cardiomegaly_docs = text_splitter.split_documents(cardiomegaly_docs)
pleuraleffusion_pneumothorax_docs = text_splitter.split_documents(pleuraleffusion_pneumothorax_docs)
tuberculosis_docs = text_splitter.split_documents(tuberculosis_docs)
xraydictionary_docs = text_splitter.split_documents(xraydictionary_docs)

In [ ]:
# Embedding Model (HuggingFace's Cloud Embedding Model)
from langchain_huggingface import HuggingFaceEndpointEmbeddings
import os
from dotenv import load_dotenv
load_dotenv()

embeddings = HuggingFaceEndpointEmbeddings(
    model = "sentence-transformers/all-MiniLM-L6-v2",
    huggingfacehub_api_token=os.getenv("HF_TOKEN")
)

In [ ]:
# Vector Store using FAISS
from langchain_community.vectorstores import FAISS
import os

os.makedirs("./faiss_db", exist_ok=True)

cardiomegaly_vector_collection = FAISS.from_documents(
    documents=cardiomegaly_docs,
    embedding=embeddings
)
cardiomegaly_vector_collection.save_local("./faiss_db/cardiomegaly")

pleuraleffusion_pneumothorax_vector_collection = FAISS.from_documents(
    documents=pleuraleffusion_pneumothorax_docs,
    embedding=embeddings
)
pleuraleffusion_pneumothorax_vector_collection.save_local("./faiss_db/pleuraleffusion_pneumothorax")

tuberculosis_vector_collection = FAISS.from_documents(
    documents=tuberculosis_docs,
    embedding=embeddings
)
tuberculosis_vector_collection.save_local("./faiss_db/tuberculosis")

xraydictionary_vector_collection = FAISS.from_documents(
    documents=xraydictionary_docs,
    embedding=embeddings
)
xraydictionary_vector_collection.save_local("./faiss_db/xraydictionary")

In [ ]:
# Loading Vector Store For Similarity Search Testing
from langchain_community.vectorstores import FAISS

# Loading the FAISS indexes
cardiomegaly_vector_collection = FAISS.load_local(
    "./faiss_db/cardiomegaly",
    embeddings,
    allow_dangerous_deserialization=True
)

pleuraleffusion_pneumothorax_vector_collection = FAISS.load_local(
    "./faiss_db/pleuraleffusion_pneumothorax",
    embeddings,
    allow_dangerous_deserialization=True
)

tuberculosis_vector_collection = FAISS.load_local(
    "./faiss_db/tuberculosis",
    embeddings,
    allow_dangerous_deserialization=True
)

xraydictionary_vector_collection = FAISS.load_local(
    "./faiss_db/xraydictionary",
    embeddings,
    allow_dangerous_deserialization=True
)

# Test query
results = xraydictionary_vector_collection.similarity_search(
    "What is cardiomegaly ?",
    k=3
)
print(results[0].page_content)

Consider this example: 
SELECT Ename, Eid, Ephone 
  FROM Employee 
  WHERE Did /H11005 15 
 This query returns the Ename, Eid, and Ephone fields from the Employee 
table for all employees assigned to department 15. 
 The view in  Figure   5.3b    is created using the following SQL statement: 
CREATE VIEW newtable (Dname, Ename, Eid, Ephone) 
AS SELECT D.Dname E.Ename, E.Eid, E.Ephone 
FROM Department D Employee E 
WHERE E.Did /H11005 D.Did 
 The preceding are just a few examples of SQL functionality. SQL statements 
can be used to create tables, insert and delete data in tables, create views, and 
retrieve data with query statements.   
5.4 DATABASE ACCESS CONTROL 
 Commercial and open-source DBMSs typically provide an access control capability 
for the database. The DBMS operates on the assumption that the computer system 
has authenticated each user. As an additional line of defense, the computer system


# Some More Cloud Embedding Techniques

In [13]:
# Embedding Model (Google's Cloud Embedding Model)
from langchain_google_genai import GoogleGenerativeAIEmbeddings
import os
from dotenv import load_dotenv
load_dotenv()

embeddings = GoogleGenerativeAIEmbeddings(
    api_key = os.getenv("GEMINI_API_KEY"),
    model="models/gemini-embedding-001"
)
vector = embeddings.embed_query("hello, world!")
vector[:5]

[-0.023955047, 0.011876456, -0.0033613679, -0.0584139, 0.0015592978]

In [ ]:
# Embedding Model (Cohere's Cloud Embedding Model)
from langchain_cohere import CohereEmbeddings
import os
from dotenv import load_dotenv
load_dotenv()

embeddings = CohereEmbeddings(
    cohere_api_key=os.getenv("COHERE_API_KEY"),
    model="embed-english-light-v3.0"
)
vector = embeddings.embed_query("hello, world!")
vector[:5]